# 02 Prepare: DICOM series to study volumes

Every later stage reads the knee MRI as one fixed-size array per study instead of the DICOM files. This notebook builds that array for the 4,407 training studies: five series per study, 72 slices in all, each slice turned to a canonical orientation, cropped to 140 mm around its centre and resized to 224 px.

It runs on CPU in about 40 minutes. Its outputs, `train_v3.npy` (about 16 GB) and the slot table `train_v3.csv`, are inputs of every training and submission notebook, and the submission notebook runs the same code on the test studies.

Imports, the competition data path, the finding names and settings shared with the training notebooks; only the data path and the image size matter here. The DICOM decoders for JPEG-compressed series are installed offline from wheels saved by the rsna-wheels kernel.

In [1]:
import glob, math, os, subprocess, sys, time
from functools import partial
from multiprocessing import Pool
import cv2, numpy as np, pandas as pd, pydicom, timm, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

R = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
       "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
S, RES = 224, 224
ARCH = "resnet50"
FOLDS, SEED = 5, 42
RUN_FOLDS = [0, 1, 2, 3, 4]
EPOCHS, BS = 14, 16
LR_BB, LR_HEAD = 1e-4, 1e-3
WIN = [5, 2, 4, 2, 3]
EVAL_DRAWS = 0
EVAL_CACHED = False
LIMIT = 0
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cuda":
    print(torch.cuda.get_device_name(0), torch.cuda.device_count())


def find(name):
    hits = [p for top in ("notebooks", "datasets", "models") for p in glob.glob(f"/kaggle/input/{top}/**/{name}", recursive=True)]
    return sorted(set(hits + glob.glob(f"/kaggle/input/*/{name}") + glob.glob(f"/kaggle/input/*/*/{name}")))


for w in find("wheels"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", w, "pylibjpeg",
                    "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm"], check=False)

ERROR: Could not find a version that satisfies the requirement pylibjpeg-libjpeg (from versions: none)
ERROR: No matching distribution found for pylibjpeg-libjpeg


A short overview of the training data, the same numbers as in notebook 01.

In [2]:
tr = pd.read_csv(f"{R}/train.csv", encoding_errors="replace")
trs = pd.read_csv(f"{R}/train_series.csv")
gold = tr[tr[LAB[0]].notna()]
print(f"{len(tr)} studies, {len(gold)} with gold labels, {len(trs)} series")
print(gold[LAB].mean().round(2).to_string())
print(trs.Anatomical_Plane.value_counts().to_dict(), trs.groupby("StudyInstanceUID").size().describe()[["mean", "max"]].to_dict())
print(tr.Report.str.len().describe()[["mean", "50%", "max"]].round().to_dict())

4407 studies, 58 with gold labels, 24371 series
ACL                 0.41
MCL                 0.16
Medial Meniscus     0.45
Lateral Meniscus    0.40
Medial OA           0.26
Lateral OA          0.19
PF OA               0.36
Effusion            0.60
Synovitis           0.47
Baker's             0.21
Contusion           0.33
Fracture            0.31
{'Sagittal': 9864, 'Coronal': 8609, 'Axial': 5898} {'mean': 5.530065804402088, 'max': 14.0}
{'mean': 1098.0, '50%': 977.0, 'max': 4743.0}


**Conclusion.** 4,407 studies, 58 with radiologist labels, 24,371 series, 5.5 series per study on average; reports average 1,098 characters.

The volume builder. For each study it fills five slots: sagittal fluid-sensitive (20 slices), sagittal T1 (12), coronal fluid-sensitive (16), coronal T1 (10) and axial fluid-sensitive (14). It prefers fat-suppressed series, never uses one series twice, and falls back to any series of the plane when a fluid-sensitive one is missing. Slices are sorted by image position along the plane's axis and sampled evenly between 4% and 96% of the series, which skips the outermost slices. Each slice is rotated and flipped to a canonical orientation from its direction cosines, cropped to 140 mm using the pixel spacing and resized to 224 px; each slot is then scaled to 8 bits between its 1st and 99.5th intensity percentiles. `load_rows` is used later: it mirrors left knees so that every knee looks like a right knee.

In [3]:
SLOTS = [("Sagittal", 1, 20), ("Sagittal", 0, 12), ("Coronal", 1, 16), ("Coronal", 0, 10), ("Axial", 1, 14)]
NS = sum(k for _, _, k in SLOTS)
FOV_MM = 140
PREP = "v3"
AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}
TARGET = {"Sagittal": ((0, 1, 0), (0, 0, -1)), "Coronal": ((1, 0, 0), (0, 0, -1)), "Axial": ((1, 0, 0), (0, 1, 0))}


def canon_op(r, c, plane):
    tr, tc = (np.array(v, float) for v in TARGET[plane])
    best = None
    for t in (0, 1):
        for fu in (0, 1):
            for fl in (0, 1):
                r1, c1 = (c, r) if t else (r, c)
                score = (-c1 if fu else c1) @ tc + (-r1 if fl else r1) @ tr
                if best is None or score > best[0]:
                    best = (score, t, fu, fl)
    return best[1:]


def canonicalise(a, op):
    t, fu, fl = op
    a = a.T if t else a
    a = a[::-1] if fu else a
    return a[:, ::-1] if fl else a


def crop_square(a, ps_r, ps_c, s=S, fov=FOV_MM):
    hp, wp = max(int(round(fov / ps_r)), 8), max(int(round(fov / ps_c)), 8)
    h, w = a.shape
    y0, x0 = (h - hp) // 2, (w - wp) // 2
    sy0, sx0, sy1, sx1 = max(y0, 0), max(x0, 0), min(y0 + hp, h), min(x0 + wp, w)
    out = np.zeros((hp, wp), np.float32)
    out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = a[sy0:sy1, sx0:sx1]
    return cv2.resize(out, (s, s), interpolation=cv2.INTER_AREA if max(hp, wp) > s else cv2.INTER_LINEAR)


def load_slot(path, plane, k):
    files = glob.glob(os.path.join(path, "*.dcm"))
    hdrs = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
    h0 = hdrs[0]
    iop = np.array(h0.ImageOrientationPatient, float)
    ps = [float(x) for x in getattr(h0, "PixelSpacing", [FOV_MM / max(int(h0.Rows), 1)] * 2)]
    try:
        key = [float(h.ImagePositionPatient[AXIS[plane]]) for h in hdrs]
    except Exception:
        key = [float(getattr(h, "InstanceNumber", i)) for i, h in enumerate(hdrs)]
    order = np.argsort(key)
    idx = order[np.linspace(0.04 * (len(order) - 1), 0.96 * (len(order) - 1), k).round().astype(int)]
    op = canon_op(iop[:3], iop[3:], plane)
    ps_r, ps_c = (ps[1], ps[0]) if op[0] else (ps[0], ps[1])
    vol = np.stack([crop_square(canonicalise(pydicom.dcmread(files[i]).pixel_array.astype(np.float32), op), ps_r, ps_c) for i in idx])
    lo, hi = np.percentile(vol, [1, 99.5])
    return (np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)


def assign_slots(d):
    used, out = set(), []
    for plane, fluid, _ in SLOTS:
        c = d[(d.Anatomical_Plane == plane) & ~d.SeriesInstanceUID.isin(used)]
        pick = c[c.Fluid_Sensitive == fluid]
        if len(pick) == 0 and fluid == 1:
            pick = c
        if len(pick) == 0:
            out.append(None)
            continue
        sid = pick.sort_values("Fat_Suppression", ascending=False, kind="stable").SeriesInstanceUID.iloc[0]
        used.add(sid)
        out.append(sid)
    return out


def _study(root, a):
    study, sids = a
    vol = np.zeros((NS, S, S), np.uint8)
    o = 0
    for (plane, _, k), sid in zip(SLOTS, sids):
        if sid is not None:
            try:
                vol[o:o + k] = load_slot(f"{root}/{study}/{sid}", plane, k)
            except Exception as e:
                print("fail", study, sid, str(e)[:80], flush=True)
                sids = [None if x == sid else x for x in sids]
        o += k
    return vol, sids


def build_volumes(ts, root, out, studies=None):
    groups = dict(tuple(ts.groupby("StudyInstanceUID")))
    studies = studies or sorted(groups)
    jobs = [(s, assign_slots(groups[s]) if s in groups else [None] * len(SLOTS)) for s in studies]
    arr = np.lib.format.open_memmap(out, "w+", np.uint8, (len(jobs), NS, S, S))
    rows = []
    with Pool(4) as p:
        for i, (v, sids) in enumerate(p.imap(partial(_study, root), jobs, chunksize=4)):
            arr[i] = v
            rows.append({"StudyInstanceUID": jobs[i][0], **{f"sid{j}": x for j, x in enumerate(sids)}})
    arr.flush()
    tab = pd.DataFrame(rows)
    tab.to_csv(out.replace(".npy", ".csv"), index=False)
    return tab


cached = find(f"train_{PREP}.npy")
if cached:
    TRV = np.load(cached[0], mmap_mode="r")
    trtab = pd.read_csv(cached[0].replace(".npy", ".csv"))
else:
    trtab = build_volumes(trs, f"{R}/train_series", f"/kaggle/working/train_{PREP}.npy")
    TRV = np.load(f"/kaggle/working/train_{PREP}.npy", mmap_mode="r")
STUDIES = list(trtab.StudyInstanceUID)
print(TRV.shape, trtab[[f"sid{j}" for j in range(len(SLOTS))]].notna().mean().round(2).to_dict())
SLOT_OFF = np.cumsum([0] + [k for _, _, k in SLOTS])


def load_rows(arr, lat, rows):
    out = np.ascontiguousarray(arr[np.asarray(rows)])
    for n, i in enumerate(rows):
        if lat[i] == 1:
            for j in (0, 1):
                out[n, SLOT_OFF[j]:SLOT_OFF[j + 1]] = out[n, SLOT_OFF[j]:SLOT_OFF[j + 1]][::-1]
            out[n, SLOT_OFF[2]:] = out[n, SLOT_OFF[2]:, :, ::-1]
    return out

(4407, 72, 224, 224) {'sid0': 1.0, 'sid1': 0.97, 'sid2': 1.0, 'sid3': 0.74, 'sid4': 1.0}


**Conclusion.** The array holds 4,407 studies x 72 slices x 224 x 224 pixels. The sagittal fluid-sensitive, coronal fluid-sensitive and axial slots are filled in effectively every study; 97% of studies have a sagittal T1 and 74% a coronal T1. A missing slot stays zero and the model masks it out.